# 🎓 AI Camera — Student Detection Training

**Simple 3-step flow:**
1. ▶️ Run **Step 1** → Upload your zip + optional previous model
2. ▶️ Run **Step 2** → Training starts automatically
3. ▶️ Run **Step 3** → Download your trained model

> ⚠️ Before starting: **Runtime → Change runtime type → T4 GPU**

In [ ]:
# ═══════════════════════════════════════════════════
# STEP 1 — Upload files & prepare dataset
# ═══════════════════════════════════════════════════
import os, zipfile, shutil, random
from google.colab import files

# Install YOLO silently
os.system('pip install ultralytics -q')

# ── 1A: Upload your dataset zip ────────────────────
print('📦 Upload your dataset zip (e.g. classroom-students_v1.yolov8.zip)')
print('   ↳ Get this from Roboflow → Export → YOLOv8 format')
print()
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

# Extract directly to /content/dataset (clean any old run first)
if os.path.exists('/content/dataset'):
    shutil.rmtree('/content/dataset')
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/dataset')
os.remove(zip_name)  # remove zip, keep things clean

# ── Fix data.yaml paths to point to /content/dataset ──
yaml_path = '/content/dataset/data.yaml'
with open(yaml_path, 'r') as f:
    content = f.read()
content = content.replace('train: ../train/images', 'train: /content/dataset/train/images')
content = content.replace('val: ../valid/images',   'val: /content/dataset/valid/images')
content = content.replace('test: ../test/images',   'test: /content/dataset/test/images')
with open(yaml_path, 'w') as f:
    f.write(content)

# ── Auto-fix: Create valid/ if Roboflow didn't include one ──
TRAIN_IMG = '/content/dataset/train/images'
TRAIN_LBL = '/content/dataset/train/labels'
VALID_IMG  = '/content/dataset/valid/images'
VALID_LBL  = '/content/dataset/valid/labels'

if not os.path.exists(VALID_IMG) or len(os.listdir(VALID_IMG)) == 0:
    print('⚠️  No validation folder found — auto-creating 20% split from train...')
    os.makedirs(VALID_IMG, exist_ok=True)
    os.makedirs(VALID_LBL, exist_ok=True)
    all_imgs = sorted([f for f in os.listdir(TRAIN_IMG) if f.endswith('.jpg')])
    random.shuffle(all_imgs)
    val_imgs = all_imgs[:max(1, int(len(all_imgs) * 0.2))]
    for img in val_imgs:
        shutil.move(os.path.join(TRAIN_IMG, img), os.path.join(VALID_IMG, img))
        lbl = img.replace('.jpg', '.txt')
        if os.path.exists(os.path.join(TRAIN_LBL, lbl)):
            shutil.move(os.path.join(TRAIN_LBL, lbl), os.path.join(VALID_LBL, lbl))

train_count = len(os.listdir(TRAIN_IMG))
val_count   = len(os.listdir(VALID_IMG))

print(f'\n✅ Dataset ready — {train_count} train  |  {val_count} val images')

# ── 1B: Upload previous model (optional) ───────────
print()
print('🤖 Do you have a previously trained model to continue from?')
answer = input('   Type  yes  to upload it, or press Enter to skip: ').strip().lower()

START_MODEL = 'yolov8n.pt'  # default: base model

if answer == 'yes':
    print()
    print('📤 Upload your previous .pt model file...')
    model_upload = files.upload()
    model_file = list(model_upload.keys())[0]
    shutil.move(model_file, '/content/start_model.pt')
    START_MODEL = '/content/start_model.pt'
    print(f'✅ Will continue training from: {model_file}')
else:
    print('✅ Will train from scratch using base YOLOv8 model')

print()
print('━' * 50)
print(f'  Train images : {train_count}')
print(f'  Val images   : {val_count}')
print(f'  Starting from: {START_MODEL}')
print('━' * 50)
print('✅ Ready! Run Step 2 to start training.')

In [ ]:
# ═══════════════════════════════════════════════════
# STEP 2 — Train the model
# ═══════════════════════════════════════════════════
from ultralytics import YOLO
import torch

print(f'🚀 Starting training on: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print()

model = YOLO(START_MODEL)

model.train(
    data     = '/content/dataset/data.yaml',
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,
    patience = 15,
    device   = 0 if torch.cuda.is_available() else 'cpu',
    project  = '/content',
    name     = 'training',
    exist_ok = True,
    verbose  = True,
)

print()
print('🏆 Training complete! Run Step 3 to download your model.')

In [ ]:
# ═══════════════════════════════════════════════════
# STEP 3 — Download your trained model
# ═══════════════════════════════════════════════════
import shutil
from google.colab import files

src = '/content/training/weights/best.pt'
dst = '/content/classroom_model.pt'
shutil.copy(src, dst)

print('⬇️  Downloading classroom_model.pt ...')
files.download(dst)
print()
print('✅ Done!')
print('   1. Rename the file to:  classroom_v2.pt')
print('   2. Place it in:         ai/models/classroom_v2.pt')
print('   3. Update config.py:    MODEL_PATH → classroom_v2.pt')